In [1]:
# ── BUILD SQLITE DATABASE ─────────────────────────────────────────────────────
import sqlite3
import pandas as pd
from pathlib import Path

CLEAN_DIR = Path("data/cleaned")
DB_PATH   = "data/tb_india.db"

# All cleaned files and their table names
table_map = {
    "tb_outcome_total.csv":            "tb_outcome_total",
    "tb_outcome_public.csv":           "tb_outcome_public",
    "tb_outcome_private.csv":          "tb_outcome_private",
    "tb_notification.csv":             "tb_notification",
    "tb_presumptive.csv":              "tb_presumptive",
    "tb_patient_characteristics.csv":  "tb_patient_characteristics",
    "tb_hiv.csv":                      "tb_hiv",
    "tb_diabetes.csv":                 "tb_diabetes",
    "tb_tobacco.csv":                  "tb_tobacco",
    "tb_gender.csv":                   "tb_gender",
    "tb_tribal.csv":                   "tb_tribal",
    "rhs_statewise.csv":               "rhs_statewise",
    "rhs_districtwise.csv":            "rhs_districtwise",
    "health_expenditure.csv":          "health_expenditure",
    "nfhs5.csv":                       "nfhs5",
    "nss_oop.csv":                     "nss_oop",
    "nss_barriers.csv":                "nss_barriers",
    "nss_treatment_source.csv":        "nss_treatment_source",
    "panel_model.csv":                 "panel_model",
    "panel_full.csv":                  "panel_full",
    "spec1_parsimonious.csv":          "spec1_parsimonious",
    "spec3_detection.csv":             "spec3_detection",
    "state_dashboard.csv":             "state_dashboard",
}

conn = sqlite3.connect(DB_PATH)

print("Loading tables into SQLite database...")
for filename, table_name in table_map.items():
    filepath = CLEAN_DIR / filename
    if filepath.exists():
        df = pd.read_csv(filepath)
        df.to_sql(table_name, conn, if_exists="replace", index=False)
        print(f"  ✓ {table_name:45s} {df.shape[0]:4d} rows × {df.shape[1]:2d} cols")
    else:
        print(f"  ✗ {filename} not found — skipping")

# ── CREATE DIMENSION TABLE ────────────────────────────────────────────────────
# Region classification for analytical grouping
state_regions = {
    "Jammu And Kashmir": "North", "Himachal Pradesh": "North",
    "Punjab": "North", "Chandigarh": "North", "Uttarakhand": "North",
    "Haryana": "North", "Delhi": "North", "Rajasthan": "North",
    "Uttar Pradesh": "North", "Bihar": "East", "Sikkim": "Northeast",
    "Arunachal Pradesh": "Northeast", "Nagaland": "Northeast",
    "Manipur": "Northeast", "Mizoram": "Northeast", "Tripura": "Northeast",
    "Meghalaya": "Northeast", "Assam": "Northeast", "West Bengal": "East",
    "Jharkhand": "East", "Odisha": "East", "Chhattisgarh": "Central",
    "Madhya Pradesh": "Central", "Gujarat": "West", "Maharashtra": "West",
    "Andhra Pradesh": "South", "Karnataka": "South", "Goa": "West",
    "Lakshadweep": "South", "Kerala": "South", "Tamil Nadu": "South",
    "Puducherry": "South", "Andaman And Nicobar Islands": "East",
    "Telangana": "South", "Ladakh": "North",
    "The Dadra And Nagar Haveli And Daman And Diu": "West",
}

dim_states = pd.DataFrame([
    {"StateName": k, "region": v,
     "is_UT": k in ["Delhi","Chandigarh","Lakshadweep","Ladakh",
                    "Puducherry","Andaman And Nicobar Islands",
                    "The Dadra And Nagar Haveli And Daman And Diu",
                    "Jammu And Kashmir"]}
    for k, v in state_regions.items()
])

# Get StateCode from any existing table
codes = pd.read_sql("SELECT DISTINCT StateCode, StateName FROM tb_outcome_total",
                    conn)
dim_states = dim_states.merge(codes, on="StateName", how="left")
dim_states.to_sql("dim_states", conn, if_exists="replace", index=False)
print(f"\n  ✓ {'dim_states':45s} {len(dim_states):4d} rows ×  4 cols")

conn.commit()
print(f"\nDatabase saved to: {DB_PATH}")
print(f"Total size: {Path(DB_PATH).stat().st_size / 1024:.0f} KB")

Loading tables into SQLite database...
  ✓ tb_outcome_total                               180 rows ×  8 cols
  ✓ tb_outcome_public                              216 rows ×  6 cols
  ✓ tb_outcome_private                             210 rows ×  6 cols
  ✓ tb_notification                                324 rows ×  6 cols
  ✓ tb_presumptive                                 216 rows ×  6 cols
  ✓ tb_patient_characteristics                     177 rows ×  6 cols
  ✓ tb_hiv                                         180 rows ×  5 cols
  ✓ tb_diabetes                                    180 rows ×  5 cols
  ✓ tb_tobacco                                     180 rows ×  5 cols
  ✓ tb_gender                                      108 rows × 13 cols
  ✓ tb_tribal                                      170 rows ×  6 cols
  ✓ rhs_statewise                                  144 rows ×  8 cols
  ✓ rhs_districtwise                              3874 rows ×  9 cols
  ✓ health_expenditure                             

In [2]:
# ── DEMONSTRATE DATABASE QUERIES ──────────────────────────────────────────────
conn = sqlite3.connect("data/tb_india.db")

print("=" * 60)
print("QUERY 1: Tables in database")
print("=" * 60)
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn
)
print(tables.to_string(index=False))

print("\n" + "=" * 60)
print("QUERY 2: States with success rate below 80% (2019-2022)")
print("=" * 60)
q2 = pd.read_sql("""
    SELECT StateName, Year, success_rate, ltfu_rate
    FROM tb_outcome_total
    WHERE success_rate < 80 AND Year >= 2019
    ORDER BY success_rate ASC
""", conn)
print(q2.to_string(index=False))

print("\n" + "=" * 60)
print("QUERY 3: Join TB outcomes with infrastructure fill rates")
print("=" * 60)
q3 = pd.read_sql("""
    SELECT 
        t.StateName,
        t.Year,
        t.success_rate,
        t.ltfu_rate,
        r.lab_tech_fill_rate,
        r.phc_doctor_fill_rate
    FROM tb_outcome_total t
    LEFT JOIN rhs_statewise r 
        ON t.StateCode = r.StateCode AND t.Year = r.Year
    WHERE t.Year = 2021
        AND r.lab_tech_fill_rate IS NOT NULL
    ORDER BY t.success_rate DESC
    LIMIT 10
""", conn)
print(q3.to_string(index=False))

print("\n" + "=" * 60)
print("QUERY 4: Average outcomes by region (using dimension table)")
print("=" * 60)
q4 = pd.read_sql("""
    SELECT 
        d.region,
        COUNT(DISTINCT t.StateCode) as n_states,
        ROUND(AVG(t.success_rate), 2) as avg_success,
        ROUND(AVG(t.ltfu_rate), 2)    as avg_ltfu,
        ROUND(AVG(t.death_rate), 2)   as avg_death
    FROM tb_outcome_total t
    JOIN dim_states d ON t.StateName = d.StateName
    WHERE t.Year >= 2019
    GROUP BY d.region
    ORDER BY avg_success DESC
""", conn)
print(q4.to_string(index=False))

print("\n" + "=" * 60)
print("QUERY 5: States where private LTFU > public LTFU")
print("=" * 60)
q5 = pd.read_sql("""
    SELECT 
        pub.StateName,
        ROUND(AVG(pub.ltfu_rate_public), 2)  as avg_public_ltfu,
        ROUND(AVG(pri.ltfu_rate_private), 2) as avg_private_ltfu,
        ROUND(AVG(pri.ltfu_rate_private - pub.ltfu_rate_public), 2) as gap
    FROM tb_outcome_public pub
    JOIN tb_outcome_private pri 
        ON pub.StateCode = pri.StateCode AND pub.Year = pri.Year
    GROUP BY pub.StateName
    HAVING gap > 0
    ORDER BY gap DESC
""", conn)
print(q5.to_string(index=False))

print("\n" + "=" * 60)
print("QUERY 6: Cluster summary from dashboard")
print("=" * 60)
q6 = pd.read_sql("""
    SELECT 
        "System Type",
        COUNT(*) as n_states,
        ROUND(AVG("Detection Rate"), 3) as avg_detection,
        ROUND(AVG("LTFU %"), 2)         as avg_ltfu,
        ROUND(AVG("Efficiency Score"), 1) as avg_efficiency
    FROM state_dashboard
    GROUP BY "System Type"
    ORDER BY avg_efficiency DESC
""", conn)
print(q6.to_string(index=False))

conn.close()

QUERY 1: Tables in database
                      name
                dim_states
        health_expenditure
                     nfhs5
              nss_barriers
                   nss_oop
      nss_treatment_source
                panel_full
               panel_model
          rhs_districtwise
             rhs_statewise
        spec1_parsimonious
           spec3_detection
           state_dashboard
               tb_diabetes
                 tb_gender
                    tb_hiv
           tb_notification
        tb_outcome_private
         tb_outcome_public
          tb_outcome_total
tb_patient_characteristics
            tb_presumptive
                tb_tobacco
                 tb_tribal

QUERY 2: States with success rate below 80% (2019-2022)
        StateName  Year  success_rate  ltfu_rate
      Lakshadweep  2019          64.0        0.0
          Mizoram  2019          67.0        1.4
          Manipur  2019          69.0        2.9
            Delhi  2019          71.0       